# Part A: Scan, explore, clean

## setup

In [ ]:
!pip -q install imagehash

from google.colab import drive
drive.mount('/content/drive')

import os
ZIP = '/content/drive/MyDrive/SmartSort_AI/data/garbage-classification.zip'
if not os.path.isdir('/content/data/garbage_classification'):
    !mkdir -p /content/data
    !unzip -q "{ZIP}" -d /content/data
print(os.listdir('/content/data'))

## build the file table

In [ ]:
from pathlib import Path
import pandas as pd

DATA_ROOT = Path('/content/data/garbage_classification')
FIG_DIR = Path('/content/drive/MyDrive/SmartSort_AI/report_figures')
OUT_DIR = Path('/content/drive/MyDrive/SmartSort_AI/outputs')
FIG_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

MERGE = {'brown-glass': 'glass', 'green-glass': 'glass', 'white-glass': 'glass'}

rows = []
for class_dir in sorted(p for p in DATA_ROOT.iterdir() if p.is_dir()):
    for f in sorted(class_dir.glob('*.jpg')):
        rows.append({'path': f'{class_dir.name}/{f.name}',
                     'orig_class': class_dir.name,
                     'label': MERGE.get(class_dir.name, class_dir.name)})
df = pd.DataFrame(rows)
print(len(df), 'images,', df['label'].nunique(), 'classes')

## scan every image (corrupt check, sizes, hashes)

In [ ]:
import hashlib
from PIL import Image
import imagehash
from tqdm.auto import tqdm

def scan_image(rel_path):
    full = DATA_ROOT / rel_path
    info = {'width': None, 'height': None, 'mode': None,
            'md5': None, 'phash': None, 'ok': False, 'error': ''}
    try:
        with open(full, 'rb') as fh:
            info['md5'] = hashlib.md5(fh.read()).hexdigest()   # exact-duplicate fingerprint
        with Image.open(full) as img:
            img.verify()                                       # corrupt-file check
        with Image.open(full) as img:
            info['width'], info['height'] = img.size
            info['mode'] = img.mode
            rgb = img.convert('RGB')                           # forces a full decode
            info['phash'] = str(imagehash.phash(rgb))          # near-duplicate fingerprint
        info['ok'] = True#
    except Exception as e:
        info['error'] = f'{type(e).__name__}: {e}'
    return info

scan = [scan_image(p) for p in tqdm(df['path'])]
df = pd.concat([df, pd.DataFrame(scan)], axis=1)
df.to_csv(OUT_DIR / 'scan_results.csv', index=False)
print('saved scan_results.csv')

## scan report

In [ ]:
print('unreadable files:', (~df['ok']).sum())
print(df.loc[~df['ok'], ['path', 'error']].to_string())
print('\nimage modes:\n', df['mode'].value_counts())
print('\nsize summary (px):\n', df[['width', 'height']].describe().round(0))
print('\nmost common sizes:\n', df.groupby(['width', 'height']).size().sort_values(ascending=False).head(8))

## EDA figures (saved to Drive for your report)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 1) Images per class: original 12 classes and merged 10 classes
orig_counts = df['orig_class'].value_counts()
merged_counts = df['label'].value_counts()
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for ax, counts, title in [(axes[0], orig_counts, 'Original 12 classes'),
                          (axes[1], merged_counts, 'After merging glass (10 classes)')]:
    bars = ax.bar(counts.index, counts.values, color='#4C78A8')
    ax.bar_label(bars, fontsize=8)
    ax.set_title(title)
    ax.set_ylabel('Images')
    ax.tick_params(axis='x', rotation=45)
    for lbl in ax.get_xticklabels():
        lbl.set_ha('right')
plt.tight_layout()
plt.savefig(FIG_DIR / 'eda_class_counts.png', dpi=150)
plt.show()

# 2) Sample grid: 6 random images per class (seeded)
classes = sorted(df['label'].unique())
n = 6
fig, axes = plt.subplots(len(classes), n, figsize=(n * 1.8, len(classes) * 1.8))
for r, c in enumerate(classes):
    picks = df[(df['label'] == c) & df['ok']].sample(n, random_state=42)['path']
    for k, p in enumerate(picks):
        ax = axes[r, k]
        ax.imshow(Image.open(DATA_ROOT / p).convert('RGB'))
        ax.axis('off')
        if k == 0:
            ax.set_title(c, fontsize=9, loc='left')
plt.tight_layout()
plt.savefig(FIG_DIR / 'eda_sample_grid.png', dpi=130)
plt.show()

# 3) Image size distribution
df['long_side'] = df[['width', 'height']].max(axis=1)
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(df['long_side'].dropna(), bins=40, color='#4C78A8')
ax.set_xlabel('Longest image side (px)')
ax.set_ylabel('Images')
ax.set_title('Image size distribution')
plt.tight_layout()
plt.savefig(FIG_DIR / 'eda_image_sizes.png', dpi=150)
plt.show()

## remove unreadable files and exact duplicates

In [ ]:
removed = []   # (path, reason)

# unreadable
bad = df[~df['ok']]
removed += [(p, 'unreadable') for p in bad['path']]
df = df[df['ok']].copy()

# exact duplicates (identical bytes)
exact = df[df.duplicated('md5', keep=False)]
print(f"{len(exact)} files sit in {exact['md5'].nunique()} exact-duplicate groups")

# same bytes but different merged labels = label conflict: remove every copy
labels_per_md5 = exact.groupby('md5')['label'].nunique()
conflict_md5 = labels_per_md5[labels_per_md5 > 1].index
is_conflict = df['md5'].isin(conflict_md5)
removed += [(p, 'exact_duplicate_label_conflict') for p in df.loc[is_conflict, 'path']]
df = df[~is_conflict]

# same bytes, same label: keep the first copy, remove the rest
extra = df.duplicated('md5', keep='first')
removed += [(p, 'exact_duplicate') for p in df.loc[extra, 'path']]
df = df[~extra].reset_index(drop=True)

print(len(df), 'images kept;', len(removed), 'removed')
print(pd.DataFrame(removed, columns=['path', 'reason'])['reason'].value_counts())

## find near-duplicates and build groups

In [ ]:
POP = np.array([bin(i).count('1') for i in range(256)], dtype=np.uint8)
hash_bytes = np.array([np.frombuffer(bytes.fromhex(h), dtype=np.uint8) for h in df['phash']])  # (N, 8)

N = len(df)
THRESH = 4        # max Hamming distance (of 64 bits) to call two images near-duplicates
CHUNK = 256
pairs = []
for start in tqdm(range(0, N, CHUNK)):
    block = hash_bytes[start:start + CHUNK]
    xor = block[:, None, :] ^ hash_bytes[None, :, :]
    dist = POP[xor].sum(axis=2)
    ii, jj = np.where(dist <= THRESH)
    keep = (ii + start) < jj                      # each pair once, no self-pairs
    ii, jj = ii[keep], jj[keep]
    pairs.extend(zip((ii + start).tolist(), jj.tolist(), dist[ii, jj].tolist()))
pairs_df = pd.DataFrame(pairs, columns=['i', 'j', 'dist'])
print(len(pairs_df), 'near-duplicate pairs found')
print(pairs_df['dist'].value_counts().sort_index())

# union-find: connect pairs into groups
parent = list(range(N))
def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x
for i, j in zip(pairs_df['i'], pairs_df['j']):
    ri, rj = find(int(i)), find(int(j))
    if ri != rj:
        parent[rj] = ri
df['group'] = [find(i) for i in range(N)]

sizes = df['group'].value_counts()
print('\ngroups with more than 1 image:', (sizes > 1).sum())
print('images inside such groups:', int(sizes[sizes > 1].sum()))
print('largest groups:', sizes.head(8).tolist())
cross = df.groupby('group')['label'].nunique()
print('groups that mix different labels:', int((cross > 1).sum()))

## near-duplicate examples for the report, then save

In [ ]:
cand = pairs_df[pairs_df['dist'] > 0]
cand = cand.sample(min(5, len(cand)), random_state=42)
if len(cand):
    fig, axes = plt.subplots(len(cand), 2, figsize=(5, 2.6 * len(cand)), squeeze=False)
    for r, (_, row) in enumerate(cand.iterrows()):
        for k, idx in enumerate((int(row['i']), int(row['j']))):
            ax = axes[r, k]
            ax.imshow(Image.open(DATA_ROOT / df.loc[idx, 'path']).convert('RGB'))
            ax.axis('off')
            ax.set_title(f"{df.loc[idx, 'label']} (distance {int(row['dist'])})", fontsize=8)
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'eda_near_duplicates.png', dpi=150)
    plt.show()

df.to_csv(OUT_DIR / 'clean_with_groups.csv', index=False)
pd.DataFrame(removed, columns=['path', 'reason']).to_csv(OUT_DIR / 'cleaning_log.csv', index=False)
print(len(df), 'images kept,', len(removed), 'removed. Saved to Drive/outputs.')

## score every candidate pair, and look at examples

In [ ]:
def thumb(rel_path, size=32):
    with Image.open(DATA_ROOT / rel_path) as img:
        return np.asarray(img.convert('L').resize((size, size), Image.Resampling.BILINEAR),
                          dtype=np.float32).ravel()

def corr(a, b):
    a = a - a.mean()
    b = b - b.mean()
    denom = np.sqrt((a * a).sum() * (b * b).sum())
    return float((a * b).sum() / denom) if denom > 0 else 1.0

thumbs = {}
def get_thumb(idx):
    if idx not in thumbs:
        thumbs[idx] = thumb(df.loc[idx, 'path'])
    return thumbs[idx]

pairs_df['corr'] = [corr(get_thumb(int(i)), get_thumb(int(j)))
                    for i, j in zip(pairs_df['i'], pairs_df['j'])]

print(pairs_df['corr'].describe().round(3))
print(pd.cut(pairs_df['corr'], [-1, 0.5, 0.8, 0.9, 0.95, 0.98, 1.0001]).value_counts().sort_index())

# 10 pairs spread from the lowest to the highest correlation
order = pairs_df.sort_values('corr').reset_index(drop=True)
picks = order.iloc[np.linspace(0, len(order) - 1, 10).astype(int)]
fig, axes = plt.subplots(len(picks), 2, figsize=(5, 2.4 * len(picks)), squeeze=False)
for r, (_, row) in enumerate(picks.iterrows()):
    for k, idx in enumerate((int(row['i']), int(row['j']))):
        ax = axes[r, k]
        ax.imshow(Image.open(DATA_ROOT / df.loc[idx, 'path']).convert('RGB'))
        ax.axis('off')
        ax.set_title(f"{df.loc[idx, 'label']} | corr {row['corr']:.2f} | pHash dist {int(row['dist'])}", fontsize=7)
plt.tight_layout()
plt.savefig(FIG_DIR / 'eda_near_dup_verification.png', dpi=150)
plt.show()

## regroup with the verified pairs only

In [ ]:
CORR_MIN = 0.90     # tentative; we confirm it from the figure above

good = pairs_df[pairs_df['corr'] >= CORR_MIN]
print(len(good), 'verified near-duplicate pairs out of', len(pairs_df), 'candidates')

parent = list(range(N))
for i, j in zip(good['i'], good['j']):
    ri, rj = find(int(i)), find(int(j))
    if ri != rj:
        parent[rj] = ri
df['group'] = [find(i) for i in range(N)]

sizes = df['group'].value_counts()
print('groups with more than 1 image:', (sizes > 1).sum())
print('images inside such groups:', int(sizes[sizes > 1].sum()))
print('largest groups:', sizes.head(8).tolist())

cross = df.groupby('group')['label'].nunique()
mixed = cross[cross > 1].index
print('groups that mix different labels:', len(mixed))
print(df[df['group'].isin(mixed)][['group', 'label', 'orig_class', 'path']]
        .sort_values('group').head(30).to_string(index=False))

df.to_csv(OUT_DIR / 'clean_with_groups.csv', index=False)
pairs_df.to_csv(OUT_DIR / 'near_duplicate_pairs.csv', index=False)

## (optional): look at the dataset’s trash class

In [ ]:
picks = df[df['label'] == 'trash'].sample(24, random_state=7)['path']
fig, axes = plt.subplots(4, 6, figsize=(11, 7.5))
for ax, p in zip(axes.ravel(), picks):
    ax.imshow(Image.open(DATA_ROOT / p).convert('RGB'))
    ax.axis('off')
plt.tight_layout()
plt.show()

## list the label-conflict files (for your report)

In [ ]:
scan_df = pd.read_csv(OUT_DIR / 'scan_results.csv')
log = pd.DataFrame(removed, columns=['path', 'reason']).merge(scan_df[['path', 'orig_class', 'md5']], on='path')
print(log[log['reason'] != 'exact_duplicate'].sort_values(['md5', 'path'])[['reason', 'orig_class', 'path']].to_string(index=False))

# Part B: the stratified 70/15/15 split

## create the split

In [ ]:
from sklearn.model_selection import train_test_split

SEED = 42

def make_split(df, seed=42):
    # one row per group; a group's label = its most common label (no mixed groups after Cell 10)
    groups = (df.groupby('group')['label']
                .agg(lambda s: s.value_counts().index[0])
                .rename('label').reset_index())
    train_g, temp_g = train_test_split(groups, test_size=0.30,
                                       stratify=groups['label'], random_state=seed)
    val_g, test_g = train_test_split(temp_g, test_size=0.50,
                                     stratify=temp_g['label'], random_state=seed)
    split_of = {}
    for name, part in [('train', train_g), ('val', val_g), ('test', test_g)]:
        split_of.update({g: name for g in part['group']})
    return df['group'].map(split_of)

df['split'] = make_split(df, SEED)

summary = pd.crosstab(df['label'], df['split'])[['train', 'val', 'test']]
summary['total'] = summary.sum(axis=1)
print(summary)
print('\noverall proportions:')
print((summary[['train', 'val', 'test']].sum() / summary['total'].sum()).round(3))
print('\nper-class proportions:')
print(summary[['train', 'val', 'test']].div(summary['total'], axis=0).round(3))

## leakage checks

In [ ]:
sp = df['split'].to_numpy()

# 1) verified near-duplicate pairs must sit in one split
apart = [(i, j) for i, j in zip(good['i'], good['j']) if sp[int(i)] != sp[int(j)]]
print('verified near-duplicate pairs split apart:', len(apart))

# informational: similar-but-different pairs (corr 0.5 to 0.9) may legitimately straddle splits
loose = pairs_df[(pairs_df['corr'] >= 0.5) & (pairs_df['corr'] < CORR_MIN)]
print('similar-but-different pairs in different splits:',
      sum(sp[int(i)] != sp[int(j)] for i, j in zip(loose['i'], loose['j'])), 'of', len(loose))

assert df.groupby('group')['split'].nunique().max() == 1, 'a group spans two splits'
assert df.groupby('md5')['split'].nunique().max() == 1, 'identical file in two splits'
assert df['path'].is_unique, 'duplicate path'
assert make_split(df, SEED).equals(df['split']), 'split is not reproducible'
print('ALL LEAKAGE CHECKS PASSED')

## class imbalance and split figure

In [ ]:
train_counts = df[df['split'] == 'train']['label'].value_counts()
K = len(train_counts)
weights = (train_counts.sum() / (K * train_counts)).round(3)
print('training images per class:\n', train_counts.to_string())
print('\nimbalance ratio (largest / smallest):', round(train_counts.max() / train_counts.min(), 2))
print('\ninverse-frequency class weights:\n', weights.to_string())

ax = summary[['train', 'val', 'test']].plot(kind='bar', stacked=True, figsize=(10, 5),
                                            color=['#4C78A8', '#F58518', '#54A24B'])
ax.set_ylabel('Images')
ax.set_title('Images per class in each split')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(FIG_DIR / 'eda_split_distribution.png', dpi=150)
plt.show()

## save the split file

In [ ]:
out = df[['path', 'label', 'split', 'orig_class']].sort_values('path')
out.to_csv(OUT_DIR / 'split_v1.csv', index=False)

chk = pd.read_csv(OUT_DIR / 'split_v1.csv')
print(chk.shape, chk.columns.tolist())
print(chk['split'].value_counts().to_dict())
print(chk.head(3).to_string(index=False))

# Part D: transforms

## load your code from GitHub and get the normalisation:

In [ ]:
!rm -rf /content/PT22026
!git clone -q --branch smartsort-ans082 --single-branch https://github.com/ansari-ai-dev/PT22026.git /content/PT22026

import sys
sys.path.insert(0, '/content/PT22026/ans082/src')
from data import get_normalization, build_transforms, load_rgb

mean, std = get_normalization('google/vit-base-patch16-224')
print('mean', mean, 'std', std)
train_tf, eval_tf = build_transforms(mean, std)

## apply the transforms to 4 training images and save the figure:

In [ ]:
import torch

split = pd.read_csv(OUT_DIR / 'split_v1.csv')
rows = split[split['split'] == 'train'].sample(4, random_state=1)

def denorm(t):
    m = torch.tensor(mean).view(3, 1, 1)
    s = torch.tensor(std).view(3, 1, 1)
    return (t * s + m).clamp(0, 1).permute(1, 2, 0).numpy()

fig, axes = plt.subplots(len(rows), 5, figsize=(12, 2.6 * len(rows)))
for r, (_, row) in enumerate(rows.iterrows()):
    img = load_rgb(DATA_ROOT / row['path'])
    axes[r, 0].imshow(img)
    axes[r, 0].set_title(f"{row['label']} (original)", fontsize=8)
    axes[r, 1].imshow(denorm(eval_tf(img)))
    axes[r, 1].set_title('eval transform', fontsize=8)
    for k in range(3):
        axes[r, 2 + k].imshow(denorm(train_tf(img)))
        axes[r, 2 + k].set_title(f'train augmentation {k + 1}', fontsize=8)
    for ax in axes[r]:
        ax.axis('off')
plt.tight_layout()
plt.savefig(FIG_DIR / 'augmentation_examples.png', dpi=130)
plt.show()

x = train_tf(img)
print('train tensor:', tuple(x.shape), x.dtype, 'min', round(float(x.min()), 2), 'max', round(float(x.max()), 2))